In [5]:
import pandas as pd
import random
import re
import os
import json
from tqdm import tqdm
import hashlib


In [3]:
# list files in /Users/ataylor/Downloads/ch_metadata_agent/htan_data


directory = '/Users/ataylor/Downloads/ch_metadata_agent/htan_data'
files = [f for f in os.listdir(directory)]
print(len(files))

716


In [49]:
# check if any files have the same md5 checksum
md5_dict = {}
for file in files:
    if file.endswith('.md5'):
        with open(os.path.join(directory, file), 'r') as f:
            md5 = f.read().strip().split()[0]
            if md5 in md5_dict:
                md5_dict[md5].append(file)
            else:
                md5_dict[md5] = [file]

duplicates = {k: v for k, v in md5_dict.items() if len(v) > 1}
print(f"Found {len(duplicates)} duplicate files based on md5 checksum.")

Found 0 duplicate files based on md5 checksum.


In [50]:
# Check if any files have the same size
size_dict = {}
for file in files: 
    if not file.endswith('.md5'):
        size = os.path.getsize(os.path.join(directory, file))
        if size in size_dict:
            size_dict[size].append(file)
        else:
            size_dict[size] = [file]
duplicates_size = {k: v for k, v in size_dict.items() if len(v) > 1}
print(f"Found {len(duplicates_size)} duplicate files based on file size.")

Found 74 duplicate files based on file size.


In [51]:
# Check if any files have same content
import hashlib
content_dict = {}
for file in files:
    if not file.endswith('.md5'):
        with open(os.path.join(directory, file), 'rb') as f:
            content = f.read()
            content_hash = hashlib.md5(content).hexdigest()
            if content_hash in content_dict:
                content_dict[content_hash].append(file)
            else:
                content_dict[content_hash] = [file]
duplicates_content = {k: v for k, v in content_dict.items() if len(v) > 1}
print(f"Found {len(duplicates_content)} duplicate files based on file content.")

Found 51 duplicate files based on file content.


In [52]:
duplicates_content

{'55ec409683dbc1a4b0c8d384cee09de3': ['syn61375759.csv',
  'syn61375798.csv',
  'syn61375776.csv',
  'syn61786218.csv',
  'syn61375761.csv',
  'syn61375756.csv',
  'syn61375807.csv',
  'syn61375754.csv'],
 '9b7673a467d1e44aa4357a80b223f4b6': ['syn24988814.txt',
  'syn24988812.txt',
  'syn24988813.txt'],
 'e2afd38f3c231b8105e24e8d7fb40b3e': ['syn62060680.csv',
  'syn62060697.csv',
  'syn62060683.csv',
  'syn62060669.csv',
  'syn62060696.csv',
  'syn62060679.csv',
  'syn62060676.csv',
  'syn62060702.csv',
  'syn62060703.csv',
  'syn62060677.csv',
  'syn62060700.csv',
  'syn62060674.csv',
  'syn62060670.csv',
  'syn62060671.csv',
  'syn62060667.csv',
  'syn62060673.csv',
  'syn62060666.csv',
  'syn62060699.csv'],
 '55e79d4c1384a3e138146e89414c8a24': ['syn24988828.txt',
  'syn24988821.txt',
  'syn24988822.txt',
  'syn24988823.txt'],
 'd8be25985e8a7ab1b4ee71dfa1d6d934': ['syn62060657.csv',
  'syn62060737.csv',
  'syn62060654.csv',
  'syn62060651.csv',
  'syn62060645.csv',
  'syn62060652.csv

In [6]:
# Based on the deduplication above write a function to parse each tsv or csv in the dict 
# into json with the file hash as the key

# the json should be flat and therefore have
# row_hash, row_content as dict, and file_paths as list
# and be deduiplicated based on row content

# we can also drp these keys
# "Component": "ImagingLevel2", "Filename": "mxif_level_2/AFsubtracted/HTA11_3252_20000010115220260050000000000.tif", "File Format": "tif", "HTAN Participant ID": "HTA11_3252", "HTAN Parent Biospecimen ID": "HTA11_3252_2000001011", "HTAN Data File ID": "HTA11_3252_20000010115220260050000000000", "Channel Metadata Filename": "mxif_level_2/AFsubtracted/metadata/HTA11_3252_20000010115220260050000000000_metadata.csv", "Imaging Assay Type": "MxIF", "Protocol Link": "NONE", "Workflow Start Datetime": "09/29/2020", "Workflow End Datetime": "09/30/2020", "Software and Version": "ImageApp 1.0.0.0", "Microscope": "INCELL ANALYZER 2500 HS", "Objective": "NIKON MRD00205 Plan Apochromat", "NominalMagnification": "20X", "LensNA": 0.75, "WorkingDistance": 1, "WorkingDistanceUnit": "mm", "Immersion": "Air", "Pyramid": "No", "Zstack": "Yes", "Tseries": "No", "Passed QC": "Yes", "Comment": NaN, "FOV number": NaN, "FOVX": NaN, "FOVXUnit": NaN, "FOVY": NaN, "FOVYUnit": NaN, "Frame Averaging": NaN, "Image ID": "MAP03252_0000_06_04_005\\AFR\\MAP03252_0000_06_04_005_ERBB2_AFR.tif", "DimensionOrder": "XYCZT", "PhysicalSizeX": 0.325, "PhysicalSizeXUnit": "µm", "PhysicalSizeY": 0.325, "PhysicalSizeYUnit": "µm", "PhysicalSizeZ": 0, "PhysicalSizeZUnit": "µm", "Pixels BigEndian": false, "PlaneCount": 0, "SizeC": 26, "SizeT": 0, "SizeX": 9375, "SizeY": 9402, "SizeZ": 0, "PixelType": "uint16", "LEVEL": "2-Processed", "TYPE": "3-AFsubtracted", "LAYERS": 26, "REGION": 5, "POSITION": 0, "LAYER": 12, "ROUND": 17, 


def parse_metadata_to_json(directory):
    result = []
    seen_rows = {}

    for file in files:
        if file.endswith('.tsv') or file.endswith('.csv'):
            file_path = os.path.join(directory, file)
            # regex match syn\d+ to get file_synid
            file_synid = None
            match = re.search(r'syn\d+', file)
            if match:
                file_synid = match.group(0)
            
            try:
                if file.endswith('.tsv'):
                    df = pd.read_csv(file_path, sep='\t', encoding='utf-8')
                else:
                    df = pd.read_csv(file_path, encoding='utf-8')
            except UnicodeDecodeError:
                try:
                    if file.endswith('.tsv'):
                        df = pd.read_csv(file_path, sep='\t', encoding='latin1')
                    else:
                        df = pd.read_csv(file_path, encoding='latin1')
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")
                    continue

            for _, row in df.iterrows():
                row_content = row.to_dict()
                # drop the keys we don't need see above
                keys_to_drop = [
                    "Component",
                    "Filename",
                    "File Format", 
                    "HTAN Participant ID", 
                    "HTAN Parent Biospecimen ID",
                    "HTAN Data File ID", 
                    "Channel Metadata Filename", 
                    "Imaging Assay Type", 
                    "Protocol Link", 
                    "Workflow Start Datetime",
                      "Workflow End Datetime", 
                      "Software and Version", 
                      "Microscope", 
                      "Objective", 
                      "NominalMagnification", 
                      "LensNA", 
                      "WorkingDistance",
                      "WorkingDistanceUnit",
                      "Pyramid","Zstack", "Tseries", "Passed QC", "Comment", "FOV number", 
                      "FOVX", "FOVXUnit", "FOVY", "FOVYUnit", "Frame Averaging", 
                      "Image ID",
                      "DimensionOrder", "PhysicalSizeX", "PhysicalSizeXUnit", 
                      "PhysicalSizeY", 
                      "PhysicalSizeYUnit", 
                      "PhysicalSizeZ", 
                      "PhysicalSizeZUnit",
                      "Pixels BigEndian", 
                      "PlaneCount",
                        "SizeC", "SizeT", "SizeX", "SizeY", "SizeZ", "PixelType", 
                        "LEVEL", "TYPE", "LAYERS", 
                        "REGION", "POSITION", "LAYER", "ROUND"] 
                
                for key in keys_to_drop:
                    row_content.pop(key, None)

                row_str = json.dumps(row_content, sort_keys=True)
                row_hash = hashlib.md5(row_str.encode('utf-8')).hexdigest()

                if row_hash in seen_rows:
                    # Add file path to existing entry
                    seen_rows[row_hash]['ch_synids'].append(file_synid)
                else:
                    # Create new entry
                    entry = {
                        'row_hash': row_hash,
                        'row_content': row_content,
                        'ch_synids': [file_synid]
                    }
                    seen_rows[row_hash] = entry
                    result.append(entry)

    return result

metadata_json = parse_metadata_to_json(directory)

print(f"Parsed {len(metadata_json)} unique rows from metadata files into JSON format.")


Parsed 3647 unique rows from metadata files into JSON format.


In [54]:
metadata_json[2:]  # show last 2 entries

[{'row_hash': '45803a4b40c470cccfd0f6e7d2b9090e',
  'row_content': {'Immersion': 'Air',
   'CHANNEL': 'Cy3',
   'MARKERID': 11,
   'MARKERNAME': 'Beta-Catenin',
   'AbID': nan,
   'CLONE': '12F751',
   'DYE': 'Dylight 550',
   'VENDOR': 'Vanderbilt Antibody and Protein Resource',
   'CATNUM': nan,
   'EXPOSURE(ms)': 50,
   'DILUTION': 0.18056},
  'ch_synids': ['syn25126380',
   'syn25126394',
   'syn25126425',
   'syn25126431',
   'syn25126419',
   'syn25126418',
   'syn25126430',
   'syn25126424',
   'syn25126395',
   'syn25126381',
   'syn25126397',
   'syn25126383',
   'syn25126432',
   'syn25126426',
   'syn25126368',
   'syn25126369',
   'syn25126427',
   'syn25126433',
   'syn25126382',
   'syn25126396',
   'syn25126392',
   'syn25126386',
   'syn25126379',
   'syn25126423',
   'syn25126422',
   'syn25126378',
   'syn25126387',
   'syn25126393',
   'syn25126385',
   'syn25126391',
   'syn25126420',
   'syn25126434',
   'syn25126435',
   'syn25126421',
   'syn25126409',
   'syn251

In [9]:
# sample 5 random entries
metadata_json_small = random.sample(metadata_json, 5)

In [10]:
# show mne entries that have more than one file path
[file for file in metadata_json if len(file['ch_synids']) > 1]

[{'row_hash': '7ab2d47382209674e5eddf4b7ed05fe5',
  'row_content': {'WorkingDistanceUnit': 'mm',
   'Immersion': 'Air',
   'Pyramid': 'No',
   'CHANNEL': 'UV',
   'MARKERID': 4,
   'MARKERNAME': 'DAPI',
   'AbID': nan,
   'CLONE': nan,
   'DYE': 'DAPI',
   'VENDOR': 'Sigma-Aldrich',
   'CATNUM': 'T9284',
   'EXPOSURE(ms)': 20,
   'DILUTION': 0.1},
  'ch_synids': ['syn25126380',
   'syn25126394',
   'syn25126425',
   'syn25126431',
   'syn25126419',
   'syn25126418',
   'syn25126430',
   'syn25126424',
   'syn25126395',
   'syn25126381',
   'syn25126397',
   'syn25126383',
   'syn25126432',
   'syn25126426',
   'syn25126368',
   'syn25126369',
   'syn25126427',
   'syn25126433',
   'syn25126382',
   'syn25126396',
   'syn25126392',
   'syn25126386',
   'syn25126379',
   'syn25126423',
   'syn25126422',
   'syn25126378',
   'syn25126387',
   'syn25126393',
   'syn25126385',
   'syn25126391',
   'syn25126420',
   'syn25126434',
   'syn25126435',
   'syn25126421',
   'syn25126409',
   'syn

In [7]:
# use claude batch api to take row_content and return a "canonical_marker"
from anthropic import Anthropic, transform_schema
from anthropic.types.beta.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.beta.messages.batch_create_params import Request
import dotenv
dotenv.load_dotenv()

from pydantic import BaseModel, Field
from enum import Enum

# marker type enum can be chemical_stain, blank_or_background, protein_group, protein_single, cd_marker, other_dna, other

class MarkerTypeEnum(str, Enum):
    chemical_stain = "chemical_stain"
    blank_or_background = "blank_or_background" 
    protein_group = "protein_group"
    protein_single = "protein_single"
    chemical_element = "chemical_element"
    cd_marker = "cd_marker"
    other_dna = "other_dna"
    other = "other"

class ExtractedlMarkerSchema(BaseModel):
    extracted_marker: str = Field(..., description="The most relevant canonical marker or identifier for the protein, stain or feature being imaged extracted from the metadata row content.")
    marker_type: MarkerTypeEnum = Field(..., description="The type of the canonical marker.")

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [56]:
# make a single request to test
def get_marker_classification(entry):
    prompt = f"""Given the following imaging metadata row content, extract the most relevant canonical marker or identifier and classify its type. If it looks to be 
Metadata Row Content: {json.dumps(entry['row_content'])}
Return the result in JSON format with the fields 'extracted_marker' and 'marker_type'."""
    response = client.beta.messages.create(
        model="claude-haiku-4-5",
        max_tokens=100,
        temperature=0.0,
        betas=["structured-outputs-2025-11-13"],
        messages=[{
            "role": "user",
            "content": prompt
        }],
        output_format={
            "type": "json_schema",
            "schema": transform_schema(ExtractedlMarkerSchema)
        }
    )
    return response.content[0].text

get_marker_classification(metadata_json_small[0])

'{"extracted_marker": "CK17", "marker_type": "protein_single"}'

In [13]:
import json
import time
from anthropic import Anthropic, transform_schema

client = Anthropic()

def process_all_metadata_with_batch(entries):
    # ... [Batch creation and submission logic same as before] ...
    
    # 1. Create Requests
    requests = []
    for idx, entry in enumerate(entries):
        requests.append({
            "custom_id": str(idx),
            "params": {
                "model": "claude-haiku-4-5",
                "max_tokens": 100,
                "messages": [{"role": "user", "content": json.dumps(entry['row_content'])}],
                "output_format": {
                    "type": "json_schema", 
                    "schema": transform_schema(ExtractedlMarkerSchema)
                }
            }
        })

    # 2. Submit Batch
    batch = client.beta.messages.batches.create(
        requests=requests,
        betas=["structured-outputs-2025-11-13"]
    )
    print(f"Batch {batch.id} submitted.")

    # 3. Wait for Completion
    while True:
        batch = client.beta.messages.batches.retrieve(batch.id)
        print(f"Batch status: {batch.processing_status}")
        if batch.processing_status == "ended":
            break
        time.sleep(10)

    # 4. Map Results and Capture Specific Errors
    results_map = {res.custom_id: res for res in client.beta.messages.batches.results(batch.id)}
    
    enhanced_entries = []
    
    for idx, entry in enumerate(entries):
        entry_enhanced = entry.copy()
        result_obj = results_map.get(str(idx))

        if not result_obj:
            # Case 1: The ID is completely missing from results (rare)
            entry_enhanced['marker_classification'] = {
                "error": "Batch result missing for this ID"
            }
        
        elif result_obj.result.type == "succeeded":
            # Case 2: Success - Parse the JSON content
            try:
                raw_json = result_obj.result.message.content[0].text
                entry_enhanced['marker_classification'] = json.loads(raw_json)
            except json.JSONDecodeError:
                entry_enhanced['marker_classification'] = {
                    "error": "Model returned invalid JSON",
                    "raw_output": raw_json
                }

        elif result_obj.result.type == "errored":
            # Case 3: API Error - Capture the REAL error message
            # The 'error' object typically has 'type' and 'message' fields
            api_error = result_obj.result.error
            error_message = getattr(api_error, "message", str(api_error))
            error_type = getattr(api_error, "type", "unknown_error")
            
            entry_enhanced['marker_classification'] = {
                "error": error_message,
                "error_type": error_type
            }

        else:
            # Case 4: Expired or other status
            entry_enhanced['marker_classification'] = {
                "error": f"Unexpected status: {result_obj.result.type}"
            }
            
        enhanced_entries.append(entry_enhanced)

    return enhanced_entries

enhanced_metadata = process_all_metadata_with_batch(metadata_json_small)

Batch msgbatch_01VNnahyApJRKj7p2PGU3nif submitted.
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: ended


In [14]:
metadata_df_enhanced = pd.DataFrame(enhanced_metadata)
metadata_df_enhanced.head()

,row_hash,row_content,ch_synids,marker_classification
0,5050fae1850cc007335e5d740969842b,"{'Channel ID': 'Channel:25', 'Channel Name': '...","[syn53271277, syn53271262, syn53271258, syn532...","{'extracted_marker': 'CK17', 'marker_type': 'p..."
1,af098cf3ef1e86b240c5172fc77a751c,"{'Channel ID': 'Channel:0:25', 'Channel Name':...",[syn25509688],"{'extracted_marker': 'CD57', 'marker_type': 'c..."
2,93d592dcbce6206faff54a7626347637,"{'filename': 'KB_SMMART_523_D23_C12R1_ASMA', '...",[syn69046893],"{'extracted_marker': 'alpha-SMA', 'marker_type..."
3,88c043e387cb8391d49f1f679cf8606d,{'filename': 'EC_SMMART_85529_D23_C09R1_FOXP3'...,[syn69046902],"{'extracted_marker': 'FOXP3', 'marker_type': '..."
4,bea74da731b34b1f34a672499c8b5b4e,"{'Channel ID': 'Channel:0:15', 'Channel Name':...","[syn26535411, syn26535409]","{'extracted_marker': 'CD8a', 'marker_type': 'c..."


In [15]:
enhanced_metadata = process_all_metadata_with_batch(metadata_json)

Batch msgbatch_01XmnDM4Nt2wAe1aCj1p8Asc submitted.
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: ended


In [16]:
enhanced_metadata_df_enhanced = pd.DataFrame(enhanced_metadata)
enhanced_metadata_df_enhanced['extracted_marker'] = enhanced_metadata_df_enhanced['marker_classification'].apply(lambda x: x.get('extracted_marker') if isinstance(x, dict) and 'extracted_marker' in x else None)
enhanced_metadata_df_enhanced['marker_type'] = enhanced_metadata_df_enhanced['marker_classification'].apply(lambda x: x.get('marker_type') if isinstance(x, dict) and 'marker_type' in x else None)
enhanced_metadata_df_enhanced.head()

,row_hash,row_content,ch_synids,marker_classification,extracted_marker,marker_type
0,7ab2d47382209674e5eddf4b7ed05fe5,"{'WorkingDistanceUnit': 'mm', 'Immersion': 'Ai...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'DAPI', 'marker_type': 'c...",DAPI,chemical_stain
1,9089bb7bec468d709780edd5eeb6f10d,"{'WorkingDistanceUnit': 'mm', 'Immersion': 'Ai...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'Alpha-actinin 4', 'marke...",Alpha-actinin 4,protein_single
2,08dc71fce97dacabf0f431b557129491,"{'WorkingDistanceUnit': 'mm', 'Immersion': 'Ai...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'Beta-Catenin', 'marker_t...",Beta-Catenin,protein_single
3,2c78bdff6bf12573097f9a589a474c2e,"{'WorkingDistanceUnit': 'mm', 'Immersion': 'Ai...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'CD11B', 'marker_type': '...",CD11B,cd_marker
4,0b85d9d3033dd1831c5e5c8f1f7d26ff,"{'WorkingDistanceUnit': 'mm', 'Immersion': 'Ai...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'CD20', 'marker_type': 'c...",CD20,cd_marker


In [17]:
enhanced_metadata_df_enhanced.groupby('marker_type').size()

marker_type
blank_or_background     129
cd_marker              1157
chemical_element         15
chemical_stain          240
other                    44
other_dna               189
protein_group           193
protein_single         1680
dtype: int64

In [18]:
# give 5 most common markers for each marker type
enhanced_metadata_df_enhanced.groupby('marker_type')['extracted_marker'].value_counts().groupby(level=0).head(5)

marker_type          extracted_marker   
blank_or_background  blank                  44
                     Empty                  21
                     Blank                  12
                     9999                   10
                     Blank_488               7
cd_marker            CD20                   84
                     CD45                   82
                     CD68                   79
                     CD3                    63
                     CD163                  62
chemical_element     CA1                     2
                     5'-HMC                  1
                     Au                      1
                     Au (Gold, mass 197)     1
                     C-12                    1
chemical_stain       DAPI                   81
                     hematoxylin            34
                     Hoechst 33342          17
                     hemotoxylin             8
                     DAPI2                   5
other              

In [19]:
# number of unuique extracted markers
enhanced_metadata_df_enhanced['extracted_marker'].nunique()

825

In [20]:
# number of unique extracted markers per marker type
enhanced_metadata_df_enhanced.groupby('marker_type')['extracted_marker'].nunique()

marker_type
blank_or_background     32
cd_marker              112
chemical_element        14
chemical_stain          47
other                   23
other_dna               56
protein_group           69
protein_single         524
Name: extracted_marker, dtype: int64

In [21]:
# show the unique markers inb protein_group
enhanced_metadata_df_enhanced[enhanced_metadata_df_enhanced['marker_type'] == 'protein_group']['extracted_marker'].unique()

array(['PanCK', 'HLA II', 'Collagen', 'PanCytokeratin', 'HLA-A',
       'Galectin', 'Pan-Cytokeratin', 'CK8/18', 'SMA', 'Pan Keratin',
       'T1 Collagen', 'TCF1/TCF7', 'Pan CytoKRT', 'HLAII',
       'pan-cytokeratin', 'pan cytokeratin', 'H3NUCA', 'HLA-II',
       'HLA-DR/DP/DQ', 'HLA class I', 'Pan-CK', 'Cytokeratin (Pan)',
       'DAPI1, pHH3, CK14, Ki67, CK19', 'panCK',
       'Donkey anti-Rat IgG-AF488', 'Donkey anti-Rabbit IgG-AF555',
       'Pan-cytokeratin', 'H3K4', 'H3K27ac', 'Lamin A+C', 'H3K27',
       'PanCK-SOX10', 'GP2', 'Goat anti-Rabbit IgG-AF488',
       'Donkey anti-Mouse IgG-AF647', 'Mouse IgG', 'BCA1',
       'MHC class I antigen ABC', 'Rat-IgG', 'Phospho-CDK1/2/3/5 (Tyr15)',
       'IgM', 'cytokeratin', 'TCRyd', 'HLA-ABC', 'PLAT/tPA', 'NaK/ATPase',
       'RNA polymerase II CTD (pS2)', 'FAP', 'Goat IgG',
       'Cytokeratin (pan)', 'HLA-DRA', 'H3K27me3', 'Pan Cytokeratin',
       'Histone H3.1 (K27me3)', 'Rb (pS807; pS811)', 'PAN-CK', 'aSMA',
       'HLA-DR', 'Nucl

In [22]:
# show the unique markers inb protein_group
enhanced_metadata_df_enhanced[enhanced_metadata_df_enhanced['marker_type'] == 'other_dna']['extracted_marker'].unique()

array(['DSDNA', 'DNA', 'anti DS DNA', 'dsDNA', 'DNA_1', 'DNA_2', 'DAPI',
       'DNA_3', 'DNA_4', 'DNA_5', 'DNA_6', 'DNA_7', 'DNA_9', 'DNA1',
       'DNA2', 'dna1', 'dna2', 'dna3', 'dna4', 'dna5', 'dna6', 'dna7',
       'dna8', 'dna9', 'dna10', 'dna11', 'dna12', 'dna13', 'dna14',
       'dna15', 'dna16', 'dna17', 'dna18', 'dna19', 'dna20', 'dna21',
       'dna22', 'dna23', 'dna24', 'dna25', 'dna26', 'dna27', 'dna28',
       'dna29', 'dna30', 'DNA_8', 'DNA_10', 'DNA_11', 'DNA_13', 'DNA_16',
       'DNA_17', 'DNA_18', 'DNA_19', 'DNA (6)', 'DNA (Hoechst 33342)',
       'DNA (11)'], dtype=object)

In [23]:
# show the unique markers inb protein_group


In [24]:

enhanced_metadata_df_enhanced[enhanced_metadata_df_enhanced['marker_type'] == 'chemical_stain']['extracted_marker'].unique()


array(['DAPI', 'SNA', 'hematoxylin', 'DAPI3', 'DES', 'DRAQ5',
       'hemotoxylin', 'HEM2', 'H&E', 'Syto13', 'DAPI2', 'DAPI1', 'DAPI6',
       'Autofluorescence-488nm', 'Autofluorescence-555nm',
       'Autofluorescence-647nm', 'Hoechst 33342', 'DAPI4', 'DAPI7',
       'DAPI8', 'DAPI9', 'DAPI10', 'Cy5', 'DNA', 'Alexa Fluor 488',
       'Alexa Fluor 555', 'Alexa Fluor 647', 'DAPI5',
       'DNA (Hoechst 33342)', 'Hoechst1', 'Hoechst2', 'Hoechst3',
       'Hoechst4', 'Hoechst5', 'Hoechst6', 'Hoechst7', 'Hoechst8',
       'Hoechst9', 'Hoechst10', 'Hoechst11', 'Hoechst12', 'Hoechst13',
       'TRITC', 'TRITC-AF', 'CY5-AF', 'Hoechst 2', "5'-HMC"], dtype=object)

In [25]:
enhanced_metadata_df_enhanced[enhanced_metadata_df_enhanced['marker_type'] == 'protein_single']['extracted_marker'].unique()


array(['Alpha-actinin 4', 'Beta-Catenin', 'CgA', 'Collagen', 'ERBB2',
       'FOXP3', 'Gamma-Actin', 'HLA-A', 'Lysozyme', 'Muc2', 'NaKATPase',
       'OLFM4', 'PCNA', 'pEGFR', 'p-STAT3', 'SMA', 'Sox9', 'Vimentin',
       'PDL1', 'EOMES', 'DC-LAMP', 'GrzB', 'T-bet', 'Foxp3', 'Ki67',
       'SMOC2', 'BMX', 'Alpha-SMA', 'MUC2', 'MKi67', 'AGR2', 'IGLL5',
       'CHGA', 'MYH11', 'SYNAPTOPHYSIN', 'KRT18', 'PDPN', 'CTNNA2',
       'WNT2B', 'WNT5B', 'RSPO3', 'Cox2', 'ACTG1', 'MUC5AC',
       'Na/K-ATPase', 'AQP5', 'Podoplanin', 'Hepatocyte Paraffin 1',
       'GLUT1', 'CK19', 'E-cadherin', 'MUC-2', 'SOX9', 'FoxP3', 'P21',
       'CK7', 'P16', 'CCL2', 'PAI1', 'CDH1', 'COL6A2', 'ALPHA-SMA', 'CEA',
       'VIMENTIN', 'SLC19A2', 'RUNX1', 'VCAN', 'SOX2', 'MUC1',
       'Beta-catenin', 'GLUT (D)', 'Keratin 14', 'PAI', 'DPEP1',
       'CEACAM5', 'PD-L1', 'LAG3', 'Granzyme B', 'PD-L2', 'Histone H3',
       'Vimentine', 'GLUT', 'HER2', 'ER', 'pHH3', 'CK14', 'CK5', 'PD1',
       'LamAC', 'aSMA', 'EP700Y

In [26]:
# flip enhanced_metadata_df_enhanced to be centered around the extraced_marker and have extraced_marker, marker_type and thewn a list of the row_hashes and ch_synids associated with that marker
extracted_marker_centered_json = []
grouped = enhanced_metadata_df_enhanced.groupby(['extracted_marker', 'marker_type'])
for (marker, marker_type), group in grouped:
    entry = {
        'extracted_marker': marker,
        'marker_type': marker_type,
        'row_hashes': group['row_hash'].tolist(),
        'row_contents': group['row_content'].tolist(),
        'ch_synids': [synid for sublist in group['ch_synids'].tolist() for synid in sublist]
    }
    extracted_marker_centered_json.append(entry)

extracted_marker_centered_json[0:2]  # show first 2 entries

[{'extracted_marker': '40S ribosomal protein S6',
  'marker_type': 'protein_single',
  'row_hashes': ['0b9e8f26bace3264979ce5e5a4469a2e'],
  'row_contents': [{'Channel ID': 'Channel:0:14',
    'Channel Name': '40S ribosomal protein S6',
    'Channel Passed QC': 'Yes',
    'Cycle Number': 4,
    'Sub Cycle Number': nan,
    'Target Name': '40S ribosomal protein S6',
    'Antibody Name': nan,
    'Antibody Role': 'primary',
    'RRID identifier': 'AB_10828226',
    'Fluorophore': 'Alexa Fluor 555',
    'Clone': '54D2',
    'Lot': nan,
    'Vendor': 'Cell Signaling Technology',
    'Catalog Number': '6989S',
    'Excitation Wavelength': 555,
    'Emission Wavelength': 590,
    'Excitation Bandwidth': 20,
    'Emission Bandwidth': 20,
    'Metal Isotope Element': nan,
    'Metal Isotope Mass': nan,
    'Oligo Barcode Upper Strand': nan,
    'Oligo Barcode Lower Strand': nan,
    'Dilution': '1:200',
    'Concentration': nan}],
  'ch_synids': ['syn25509678']},
 {'extracted_marker': '40S rib

In [27]:
# show an example with moe than one row hash
for entry in extracted_marker_centered_json:
    if len(entry['row_hashes']) > 1:
        print(entry)
        break

{'extracted_marker': '53BP1', 'marker_type': 'protein_single', 'row_hashes': ['ce21ca7ecf58d56f62f87b88dfcb5ad1', 'f54793dd5cb0fa907da1be6c9268c5c8'], 'row_contents': [{'ID': 'R3-5', 'Markers': '53BP1', 'Round': 3, 'Channel': 5, 'ExpTime': 400}, {'ID': 'R2-5', 'Markers': '53BP1', 'Round': 2, 'Channel': 5, 'ExpTime': 0}], 'ch_synids': ['syn52287352', 'syn52287355', 'syn52287382']}


In [28]:
# Simple Claude API call using uniprot_api as a tool
from anthropic import Anthropic
from uniprot_api import lookup_protein
import json

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Define the tool schema for uniprot_api
tools = [
    {
        "name": "lookup_protein",
        "description": "Searches the UniProt database for protein information by gene name or protein name. Returns detailed protein entries including gene name, protein name, organism, subcellular location, function, and confidence score.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The gene name or protein name to search for (e.g., 'CD20', 'FOXP3', 'Ki67')"
                },
                "organism": {
                    "type": "string",
                    "description": "The organism to filter by (default: 'human')",
                    "default": "human"
                }
            },
            "required": ["query"]
        }
    }
]

# Make a Claude API call asking about a protein
user_message = "Can you look up information about the protein CD20 in humans?"

response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    tools=tools,
    messages=[{"role": "user", "content": user_message}]
)

print("Claude's initial response:")
print(response)

# Check if Claude wants to use the tool
if response.stop_reason == "tool_use":
    # Extract the tool use request
    tool_use_block = next(block for block in response.content if block.type == "tool_use")
    
    print(f"\n🔧 Claude wants to call: {tool_use_block.name}")
    print(f"📝 With arguments: {tool_use_block.input}")
    
    # Execute the actual function
    query = tool_use_block.input["query"]
    organism = tool_use_block.input.get("organism", "human")
    
    print(f"\n🔍 Calling lookup_protein('{query}', '{organism}')...")
    results = lookup_protein(query, organism)
    
    # Format the results
    tool_result = {
        "results": [entry.to_dict() for entry in results]
    }
    
    print(f"✅ Found {len(results)} results")
    
    # Send the tool result back to Claude
    final_response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        tools=tools,
        messages=[
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": response.content},
            {
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_use_block.id,
                        "content": json.dumps(tool_result)
                    }
                ]
            }
        ]
    )
    
    print("\n📊 Claude's final response:")
    print(final_response.content[0].text)

Claude's initial response:
Message(id='msg_01R7zddLoejVgBinXzjFpDi4', content=[ToolUseBlock(id='toolu_01WteYHvizCjRB7MZKb7QP5Q', input={'query': 'CD20', 'organism': 'human'}, name='lookup_protein', type='tool_use')], model='claude-sonnet-4-5-20250929', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=674, output_tokens=71, server_tool_use=None, service_tier='standard'))

🔧 Claude wants to call: lookup_protein
📝 With arguments: {'query': 'CD20', 'organism': 'human'}

🔍 Calling lookup_protein('CD20', 'human')...
✅ Found 10 results

📊 Claude's final response:
Based on the search results, I found information about **CD20** in humans:

## Primary Match: B-lymphocyte antigen CD20

**Gene Name:** MS4A1  
**Accession:** P11836  
**Organism:** Homo sapiens (Human)

### Subcellular Location:
- Cell 

In [ ]:
# Intelligent Claude-based UniProt lookup with metadata context
from anthropic import Anthropic, transform_schema
from uniprot_api import lookup_protein
import json

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# simple schema with uniprot id and subcellular location
#   where subcellular_location is one off cytoplasm, nucleus, cell memberane, extracellular.
class SubcellularLocationEnum(str, Enum):
    cytoplasm = "cytoplasm"
    nucleus = "nucleus"
    cell_membrane = "cell_membrane"
    extracellular = "extracellular"

class UniprotOutputSchema(BaseModel):
    accession: str = Field(..., description="The UniProt accession ID of the protein.")
    gene_name: str = Field(..., description="The gene name associated with the protein.")
    protein_name: str = Field(..., description="The full name of the protein.")
    organism: str = Field(..., description="The organism from which the protein is derived.")
    subcellular_location: SubcellularLocationEnum = Field(..., description="List of subcellular locations where the protein is found.")
    #function: str = Field(..., description="A brief description of the protein's function.")
    confidence_score: float = Field(..., description="Confidence score of the protein match (0 to 1).")
    
# Define the tool - Claude calls this to search UniProt
tools = [
    {
        "name": "lookup_protein",
        "description": "Searches the UniProt database for protein information by gene name or protein name. Returns a list of matching protein entries with details like gene name, protein name, organism, subcellular location, function, and confidence score. Use this to find candidate proteins that match the marker.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The gene name or protein name to search for (e.g., 'CD20', 'FOXP3', 'Ki67')"
                },
                "organism": {
                    "type": "string",
                    "description": "The organism to filter by (default: 'human')",
                    "default": "human"
                }
            },
            "required": ["query"]
        }
    }
]

def curate_marker_with_claude(marker_entry):
    """
    Use Claude to intelligently curate a marker by:
    1. Looking up the marker in UniProt
    2. Using ALL the metadata context to select the best match
    3. Returning the selected UniProt entry with reasoning
    """
    
    extracted_marker = marker_entry['extracted_marker']
    marker_type = marker_entry['marker_type']
    row_contents = marker_entry['row_contents']
    
    # Build a rich prompt with ALL context
    user_message = f"""I need you to find the correct UniProt entry for this antibody marker used in imaging:

**Extracted Marker**: {extracted_marker}
**Marker Type**: {marker_type}

**Imaging Metadata** (use this context to select the right protein):
{json.dumps(row_contents, indent=2)}

Please:
1. Search UniProt for "{extracted_marker}"
2. Review all the results you find
3. Use the imaging metadata (antibody clone, vendor, catalog number, target name, etc.) to determine which UniProt entry is the correct match
4. If there are multiple good matches, explain why you chose one
5. If the metadata suggests this isn't actually a protein (e.g., it's a control antibody, isotype control, or chemical stain), note that

Return your analysis with:
- The selected UniProt accession and gene name
- Your confidence level (high/medium/low)
- Your reasoning for the selection"""

    # Initial call - Claude decides to use the tool
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=4096,
        tools=tools,
        messages=[{"role": "user", "content": user_message}]
    )
    
    # Check if Claude wants to use the tool
    if response.stop_reason == "tool_use":
        tool_use_block = next(block for block in response.content if block.type == "tool_use")
        
        # Execute the UniProt lookup
        query = tool_use_block.input["query"]
        organism = tool_use_block.input.get("organism", "human")
        results = lookup_protein(query, organism)
        
        # Format results for Claude
        tool_result_text = f"Found {len(results)} UniProt entries for '{query}':\n\n"
        for i, entry in enumerate(results, 1):
            tool_result_text += f"Result {i}:\n"
            tool_result_text += f"  Accession: {entry.accession}\n"
            tool_result_text += f"  Gene: {entry.gene_name}\n"
            tool_result_text += f"  Protein: {entry.protein_name}\n"
            tool_result_text += f"  Organism: {entry.organism}\n"
            tool_result_text += f"  Confidence: {entry.confidence_score}\n"
            if entry.subcellular_location:
                tool_result_text += f"  Location: {', '.join(entry.subcellular_location)}\n"
            if entry.function:
                tool_result_text += f"  Function: {entry.function[:300]}...\n"
            tool_result_text += "\n"
        
        # Continue conversation with tool results
        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": [
                {
                    "type": "tool_use",
                    "id": tool_use_block.id,
                    "name": tool_use_block.name,
                    "input": tool_use_block.input
                }
            ]},
            {
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_use_block.id,
                        "content": tool_result_text
                    }
                ]
            }
        ]
        
        # Get Claude's analysis
        final_response = client.beta.messages.create(
            model="claude-sonnet-4-5",
            betas =["structured-outputs-2025-11-13"],
            max_tokens=4096,
            tools=tools,
            messages=messages,
            output_format={ 
                    "type": "json_schema", 
                    "schema": transform_schema(UniprotOutputSchema)
            }
        )
        
        return {
            'extracted_marker': extracted_marker,
            'marker_type': marker_type,
            #'uniprot_results_count': len(results),
            'uniprot_summary': final_response.content[0].text,
            #'all_uniprot_results': [r.to_dict() for r in results]
        }
    
    else:
        # Claude responded without using tools
        return {
            'extracted_marker': extracted_marker,
            'marker_type': marker_type,
            'claude_response': response.content[0].text
        }


# Test with a real example from your data
# First, let's restructure your data to include row_contents for each unique marker
def prepare_marker_entries(df):
    """Group by extracted_marker and collect all associated metadata"""
    marker_entries = []
    
    for marker in df['extracted_marker'].unique():
        if pd.isna(marker) or marker in ['blank', 'Empty', 'Blank', '9999']:
            continue
            
        marker_rows = df[df['extracted_marker'] == marker]
        marker_type = marker_rows.iloc[0]['marker_type']
        
        # Collect all row contents for this marker
        row_contents = []
        row_hashes = []
        ch_synids = []
        
        for _, row in marker_rows.iterrows():
            row_contents.append(row['row_content'])
            row_hashes.append(row['row_hash'])
            ch_synids.extend(row['ch_synids'])
        
        marker_entries.append({
            'extracted_marker': marker,
            'marker_type': marker_type,
            'row_hashes': row_hashes,
            'row_contents': row_contents,
            'ch_synids': list(set(ch_synids))
        })
    
    return marker_entries

# Test with one marker
print("Preparing marker entries...")
marker_entries = prepare_marker_entries(enhanced_metadata_df_enhanced)
print(f"Found {len(marker_entries)} unique markers to curate")

# Test with CD20
cd20_entry = [m for m in marker_entries if m['extracted_marker'] == 'CD20'][0]
print(f"\nTesting with CD20 marker...")
print(f"  Marker type: {cd20_entry['marker_type']}")
print(f"  Number of metadata rows: {len(cd20_entry['row_contents'])}")
print(f"  Number of synapse IDs: {len(cd20_entry['ch_synids'])}")

result = curate_marker_with_claude(cd20_entry)

print("\n" + "="*80)
print("CLAUDE'S ANALYSIS")
print("="*80)
print(result)

Preparing marker entries...
Found 821 unique markers to curate

Testing with CD20 marker...
  Marker type: cd_marker
  Number of metadata rows: 84
  Number of synapse IDs: 514

CLAUDE'S ANALYSIS
{'extracted_marker': 'CD20', 'marker_type': 'cd_marker', 'uniprot_summary': '{"accession": "P11836", "gene_name": "MS4A1", "protein_name": "B-lymphocyte antigen CD20", "organism": "Homo sapiens", "subcellular_location": "cell_membrane", "confidence_score": 1.0}'}


In [30]:
# sample 5 rows from protein_single marker type
protein_single_entries = [m for m in marker_entries if m['marker_type'] == 'protein_single']
sampled_entries = random.sample(protein_single_entries, 5)

for entry in sampled_entries:
    print(f"\nCurating marker: {entry['extracted_marker']} ({len(entry['row_contents'])} metadata rows)")
    
    result = curate_marker_with_claude(entry)
    print(result['uniprot_summary'])


Curating marker: Cytokeratin 14 (1 metadata rows)
 {"accession": "P02533", "gene_name": "KRT14", "protein_name": "Keratin, type I cytoskeletal 14", "organism": "Homo sapiens", "subcellular_location": "cytoplasm", "confidence_score": 0.95}

Curating marker: MART-1 (2 metadata rows)
{"accession": "Q16655", "gene_name": "MLANA", "protein_name": "Melanoma antigen recognized by T-cells 1", "organism": "Homo sapiens", "subcellular_location": "cell_membrane", "confidence_score": 0.95}

Curating marker: cGAS (4 metadata rows)


KeyboardInterrupt: 

In [44]:
for entry in protein_single_entries:
    uniprot_results = lookup_protein(entry['extracted_marker'], organism='human')
    print(f"\nMarker: {entry['extracted_marker']} - Found {len(uniprot_results)} UniProt entries")
    # print top highest confidence score hit
    if uniprot_results:
        top_hit = max(uniprot_results, key=lambda x: x.confidence_score)
        print(f"  Top Hit: {top_hit.accession} | Gene: {top_hit.gene_name} | Protein: {top_hit.protein_name} | Confidence: {top_hit.confidence_score}")



Marker: Alpha-actinin 4 - Found 10 UniProt entries
  Top Hit: O43707 | Gene: ACTN4 | Protein: Alpha-actinin-4 | Confidence: 0.6

Marker: Beta-Catenin - Found 10 UniProt entries
  Top Hit: P35222 | Gene: CTNNB1 | Protein: Catenin beta-1 | Confidence: 0.6

Marker: CgA - Found 10 UniProt entries
  Top Hit: P01215 | Gene: CGA | Protein: Glycoprotein hormones alpha chain | Confidence: 1.0

Marker: Collagen - Found 10 UniProt entries
  Top Hit: Q02388 | Gene: COL7A1 | Protein: Collagen alpha-1(VII) chain | Confidence: 0.6

Marker: ERBB2 - Found 10 UniProt entries
  Top Hit: P04626 | Gene: ERBB2 | Protein: Receptor tyrosine-protein kinase erbB-2 | Confidence: 1.0

Marker: FOXP3 - Found 10 UniProt entries
  Top Hit: Q9BZS1 | Gene: FOXP3 | Protein: Forkhead box protein P3 | Confidence: 1.0

Marker: Gamma-Actin - Found 10 UniProt entries
  Top Hit: P63267 | Gene: ACTG2 | Protein: Actin, gamma-enteric smooth muscle | Confidence: 0.6

Marker: HLA-A - Found 10 UniProt entries
  Top Hit: P04439 | G

In [ ]:
import json
import os
import concurrent.futures
from tqdm import tqdm
from anthropic import Anthropic, transform_schema
from uniprot_api import lookup_protein # Ensure this is your actual import

# [Paste your Schema and Tool Definitions here from your previous code]
# ... SubcellularLocationEnum, UniprotOutputSchema, tools ...

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def curate_single_marker(marker_entry):
    """
    Self-contained function to process one marker.
    Includes the full conversation loop: Ask -> Tool Use -> Tool Result -> Final Answer
    """
    try:
        extracted_marker = marker_entry['extracted_marker']
        marker_type = marker_entry['marker_type']
        row_contents = marker_entry['row_contents']
        
        # 1. Build the prompt
        user_message = f"""Find the correct UniProt entry for this marker:
        **Marker**: {extracted_marker} ({marker_type})
        **Context**: {json.dumps(row_contents, indent=2)}
        
        Search UniProt, review results against metadata (clone, vendor, etc.), and identify the specific target."""

        # 2. First Call: Get Search Intent
        response = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=1024,
            tools=tools,
            messages=[{"role": "user", "content": user_message}]
        )

        # 3. Handle Tool Use
        if response.stop_reason == "tool_use":
            tool_use = next(block for block in response.content if block.type == "tool_use")
            
            # Run local Python function
            query = tool_use.input["query"]
            organism = tool_use.input.get("organism", "human")
            # print(f"Searching: {query}...") # Optional logging
            
            results = lookup_protein(query, organism)
            
            # Format results for Claude
            # Limit to top 5-10 to save context window
            tool_result_text = f"Found {len(results)} entries. Top results:\n"
            for entry in results[:7]: 
                tool_result_text += (f"- {entry.accession} ({entry.gene_name}): {entry.protein_name}. "
                                     f"Loc: {entry.subcellular_location}. Conf: {entry.confidence_score}\n")

            # 4. Final Call: Get Structured Analysis
            final_response = client.beta.messages.create(
                model="claude-haiku-4-5",
                betas=["structured-outputs-2025-11-13"],
                max_tokens=4096,
                tools=tools,
                messages=[
                    {"role": "user", "content": user_message},
                    {"role": "assistant", "content": response.content}, # Original tool use request
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "tool_result",
                                "tool_use_id": tool_use.id,
                                "content": tool_result_text
                            }
                        ]
                    }
                ],
                output_format={
                    "type": "json_schema",
                    "schema": transform_schema(UniprotOutputSchema)
                }
            )
            
            # Parse the structured output
            structured_data = json.loads(final_response.content[0].text)
            
            # Return combined data
            return {
                **marker_entry,
                "uniprot_data": structured_data,
                "status": "success"
            }
            
        else:
            # Claude didn't search (maybe it knew the answer or refused)
            return {
                **marker_entry,
                "error": "Claude did not trigger a search tool",
                "raw_response": response.content[0].text,
                "status": "no_search"
            }

    except Exception as e:
        return {
            **marker_entry,
            "error": str(e),
            "status": "failed"
        }

def run_concurrent_processing(marker_entries, max_workers=20):
    """
    Process markers in parallel threads.
    """
    results = []
    
    # ThreadPoolExecutor is perfect for I/O bound tasks like API calls
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_marker = {executor.submit(curate_single_marker, m): m for m in marker_entries}
        
        # Process as they complete (using tqdm for a progress bar)
        print(f"Processing {len(marker_entries)} markers with {max_workers} threads...")
        for future in tqdm(concurrent.futures.as_completed(future_to_marker), total=len(marker_entries)):
            data = future.result()
            results.append(data)
            
    return results

# --- Execution ---

# 1. Prepare data
# marker_entries = prepare_marker_entries(enhanced_metadata_df_enhanced)

# 2. Run
# For 513 inputs, with 10 workers, this should take about 2-5 minutes total.
# Adjust max_workers based on your Rate Limits (Tier 2/3 accounts can handle 20+)

protein_single_entries = [m for m in marker_entries if m['marker_type'] == 'protein_single']



final_results = run_concurrent_processing(protein_single_entries, max_workers=10)



Processing 490 markers with 10 threads...


 32%|███▏      | 157/490 [01:45<03:44,  1.48it/s]


In [46]:
final_results_df = pd.DataFrame(final_results)
final_results_df.error[0]  # show first 5 errors

NameError: name 'final_results' is not defined

In [ ]:
# One shot with tool use

class MarkerTypeEnum(str, Enum):
    chemical_stain = "chemical_stain"
    blank_or_background = "blank_or_background" 
    protein_group = "protein_group"
    protein_single = "protein_single"
    chemical_element = "chemical_element"
    cd_marker = "cd_marker"
    other_dna = "other_dna"
    other = "other"

class ExtractedlMarkerSchema(BaseModel):
    extracted_marker: str = Field(..., description="The most relevant canonical marker or identifier for the protein, stain or feature being imaged extracted from the metadata row content.")
    marker_type: MarkerTypeEnum = Field(..., description="The type of the canonical marker.")


# make a single request to test
def get_marker_classification(entry):
    prompt = f"""Given the following imaging metadata row content, extract the most relevant canonical marker or identifier and classify its type. If it looks to be 
Metadata Row Content: {json.dumps(entry['row_content'])}
Return the result in JSON format with the fields 'extracted_marker' and 'marker_type'."""
    response = client.beta.messages.create(
        model="claude-haiku-4-5",
        max_tokens=100,
        temperature=0.0,
        betas=["structured-outputs-2025-11-13"],
        messages=[{
            "role": "user",
            "content": prompt
        }],
        output_format={
            "type": "json_schema",
            "schema": transform_schema(ExtractedlMarkerSchema)
        }
    )
    return response.content[0].text

get_marker_classification(metadata_json_small[0])